# Metric learning: contrastive & triplet loss

_Few-shot & Transfer Learning_

**Teach a network to place same-class things close and different-class things far, then judging is just measuring distance.**

Metric learning trains a network to turn each input into a short list of numbers, called an embedding. Think of the embedding as a point on a map.

       The goal: put points of the same class close together, and points of different classes far apart.

---

This notebook is a practice scaffold for the **Metric learning: contrastive & triplet loss** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Visualize the data & results

_On real digit images, does a learned metric (NCA) separate the classes more cleanly than plain PCA?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.neighbors import NeighborhoodComponentsAnalysis  # supervised metric learning

digits = load_digits()                         # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                          # pixel intensities scaled to 0..1
y = digits.target

classes = [0, 1, 2, 3]                          # pick 4 digit classes
idx = []
for c in classes:                               # up to 15 real images per class
    idx += list(np.where(y == c)[0][:15])
idx = np.array(idx)
Xs, ys = X[idx], y[idx]                          # <= 60 total points

# unsupervised baseline: plain PCA to 2-D (no labels used)
pca = PCA(n_components=2, random_state=0).fit_transform(Xs)

# supervised metric learning: NCA learns a 2-D space where same-class points cluster
nca = NeighborhoodComponentsAnalysis(n_components=2, random_state=0).fit(Xs, ys)
emb = nca.transform(Xs)

for name, Z in [("PCA", pca), ("NCA", emb)]:    # print the real plotted coordinates
    print(name)
    for c in classes:
        pts = np.round(Z[ys == c], 2).tolist()
        print(" digit", c, pts)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Triplet loss with margin $\alpha = 0.5$. Embeddings (1-D): anchor $f(a) = 0$, positive $f(p) = 0.4$, negative $f(n) = 0.6$. Find the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the same-class squared distance $\|f(a)-f(p)\|^2 = (0 - 0.4)^2$. — _This is how far the anchor is from the same-class point; we want it small._
- Compute the different-class squared distance $\|f(a)-f(n)\|^2 = (0 - 0.6)^2$. — _This is how far the anchor is from the different-class point; we want it large._
- Plug into $\max(0,\; 0.16 - 0.36 + 0.5)$. — _Subtract the two distances and add the margin, then clip negatives to 0._

**Answer:** $\|f(a)-f(p)\|^2 = (-0.4)^2 = 0.16$. $\|f(a)-f(n)\|^2 = (-0.6)^2 = 0.36$. Inside: $0.16 - 0.36 + 0.5 = 0.30$. Loss $= \max(0, 0.30) = 0.30$. It is positive because the negative (0.6) is not far enough past the positive (0.4) to clear the 0.5 margin, so backprop still has work to do.

</details>

**Problem 2.** In plain English, what does the margin $\alpha$ do, and why does setting $\alpha = 0$ hurt?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the loss is 0 only when the different-class distance beats the same-class distance by at least $\alpha$. — _The margin is the size of the gap we insist on between classes._
- Consider what "just barely separated" means when $\alpha = 0$. — _With no gap, classes can touch, so a tiny bit of noise flips the decision._

**Answer:** The margin $\alpha$ is the minimum gap we demand between the same-class distance and the different-class distance. It forces the clusters apart with room to spare. With $\alpha = 0$ the network only has to make the negative a hair farther than the positive, so the classes can sit right against each other and a single noisy point can be misjudged.

</details>